In [29]:
# Load API keys securely from a .env file (never hardcode keys in a notebook).
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
EXCHANGE_RATE_API_KEY = os.getenv("EXCHANGE_RATE_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")
if not EXCHANGE_RATE_API_KEY:
    raise ValueError("EXCHANGE_RATE_API_KEY not found. Add it to your .env file.")

In [3]:
%pip install -U langchain-groq langchain-core langchain-classic requests python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
import os

In [5]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [6]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [7]:
multiply.name

'multiply'

In [9]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [8]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [10]:
# tool binding

In [11]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [12]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond appropriately. It\'s a simple greeting. We can respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 72, 'total_tokens': 113, 'completion_time': 0.08575708, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.002914193, 'prompt_tokens_details': None, 'queue_time': 0.292726591, 'total_time': 0.088671273}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1b5d4-4aae-772d-ab5b-68d2632dc773-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 41, 'total_tokens': 113, 'output_token_details': {'reasoning': 23}})

In [13]:
llm_with_tools = llm.bind_tools([multiply])

In [14]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm doing great, thank you for asking. How can I assist you today?", additional_kwargs={'reasoning_content': 'The user says "Hi how are you". It\'s a casual greeting. We respond politely. No need for function calls.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 133, 'total_tokens': 185, 'completion_time': 0.10750288, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.006542901, 'prompt_tokens_details': None, 'queue_time': 0.369968781, 'total_time': 0.114045781}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1b5d4-9d47-71d3-a9d1-e33a59b7cec1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 52, 'total_tokens': 185, 'output_token_details': {'reasoning': 25}})

In [15]:
query = HumanMessage('can you multiply 3 with 1000')

In [16]:
messages = [query]

In [17]:
result = llm_with_tools.invoke(messages)

In [18]:
messages.append(result)

In [19]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants multiplication of 3 and 1000. Use function multiply.', 'tool_calls': [{'id': 'fc_56faf0af-fc5a-4aab-be2f-873adc86765e', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 138, 'total_tokens': 188, 'completion_time': 0.104980883, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.005497863, 'prompt_tokens_details': None, 'queue_time': 0.157728656, 'total_time': 0.110478746}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c3da0d4bb9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1b5d5-2fc8-7478-97de-320b30ab4fad-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'fc_56faf0

In [20]:
tool_result = multiply.invoke(result.tool_calls[0])

In [21]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='fc_56faf0af-fc5a-4aab-be2f-873adc86765e')

In [22]:
messages.append(tool_result)

In [23]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants multiplication of 3 and 1000. Use function multiply.', 'tool_calls': [{'id': 'fc_56faf0af-fc5a-4aab-be2f-873adc86765e', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 138, 'total_tokens': 188, 'completion_time': 0.104980883, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.005497863, 'prompt_tokens_details': None, 'queue_time': 0.157728656, 'total_time': 0.110478746}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c3da0d4bb9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1b5d5-2fc8-7478-97de-320b30ab4fad-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'fc_56faf0

In [24]:
llm_with_tools.invoke(messages).content

'The product of\u202f3\u202fand\u202f1000\u202fis\u202f**3000**.'

In [28]:
# Currency conversion tools
from langchain_core.tools import InjectedToolArg
from typing import Annotated

EXCHANGE_RATE_API_KEY = os.getenv("EXCHANGE_RATE_API_KEY")

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> dict:
    """
    Fetch the current exchange-rate information between two currencies.
    Example: base_currency='USD', target_currency='INR'
    """
    base = base_currency.upper()
    target = target_currency.upper()

    # Use API key endpoint if key is present; otherwise fallback to the free open endpoint
    if EXCHANGE_RATE_API_KEY:
        url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/pair/{base}/{target}"
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()
    else:
        # Open access endpoint (no API key required)
        url = f"https://open.er-api.com/v6/latest/{base}"
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()
        if "rates" in data and target in data["rates"]:
            data["conversion_rate"] = data["rates"][target]
            data["base_code"] = base
            data["target_code"] = target

    if data.get("result") != "success":
        raise ValueError(f"Exchange-rate API error: {data}")

    return data


@tool
def convert(
    base_currency_value: float,
    conversion_rate: Annotated[float, InjectedToolArg]
) -> float:
    """
    Convert an amount using a conversion rate supplied by the calling code.
    """
    return base_currency_value * conversion_rate


In [30]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'number'}}

In [31]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1794009601,
 'time_last_update_utc': 'Sat, 07 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1794096001,
 'time_next_update_utc': 'Sun, 08 Nov 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.1392}

In [32]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [33]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [34]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [35]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [36]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [37]:
ai_message = llm_with_tools.invoke(messages)

In [38]:
messages.append(ai_message)

In [39]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc_0c17c88c-1d22-48ff-8d59-51ec07dccc69',
  'type': 'tool_call'}]

In [41]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [42]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': "We need to get conversion factor between INR and USD. Use function get_conversion_factor with base_currency='INR', target_currency='USD' perhaps. Then compute conversion of 10 INR to USD using that factor. Could also use convert function but that expects base_currency_value and maybe conversion factor? Actually convert takes base_currency_value only; maybe it uses previously fetched factor? Not sure. Safer: get factor, then compute manually: USD = INR * factor? Usually conversion factor from INR to USD is something like 0.012 (1 INR = 0.012 USD). We'll fetch.", 'tool_calls': [{'id': 'fc_0c17c88c-1d22-48ff-8d59-51ec07dccc69', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, 

In [43]:
llm_with_tools.invoke(messages).content

'The current exchange‑rate data shows:\n\n- **1\u202fINR ≈ 0.01062\u202fUSD**\n\nUsing that rate:\n\n- **10\u202fINR × 0.01062\u202fUSD/INR ≈ 0.1062\u202fUSD**\n\nSo, 10 Indian rupees is roughly **$0.11\u202fUSD** (rounded to two decimal places).'

In [45]:
# Modern agent example (LangChain create_agent API)
from langchain.agents import create_agent

@tool
def currency_converter(
    amount: float,
    base_currency: str,
    target_currency: str
) -> dict:
    """
    Convert an amount from one currency to another using the latest available
    exchange rate. Example: amount=10, base_currency='INR', target_currency='USD'.
    """
    base = base_currency.upper()
    target = target_currency.upper()
    api_key = os.getenv("EXCHANGE_RATE_API_KEY") or EXCHANGE_RATE_API_KEY

    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{base}/{target}"
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    data = response.json()

    if data.get("result") != "success":
        raise ValueError(f"Exchange-rate API error: {data}")

    rate = data["conversion_rate"]
    return {
        "amount": amount,
        "base_currency": base,
        "target_currency": target,
        "conversion_rate": rate,
        "converted_amount": amount * rate
    }

agent = create_agent(
    model=llm,
    tools=[currency_converter],
    system_prompt="You are a helpful assistant. Use the currency_converter tool for live currency conversions."
)


In [46]:
# Test the agent
response = agent.invoke({
    "messages": [HumanMessage("Can you convert 50 USD to INR?")]
})
print(response["messages"][-1].content)


Sure! Using the latest exchange rate:

- **Conversion rate:** 1 USD = 94.1392 INR  
- **50 USD** = **≈ 4,706.96 INR**

Let me know if you need any other conversions or additional details!
